# Évaluation finale des modèles

## Objectif

Ce notebook réalise l’évaluation finale des modèles Gemini et OpenAI sur les questions médicales MCQU du split `test`.

Le prompt sélectionné dans le notebook précédent est utilisé de manière identique pour tous les modèles. Le split `test` n’a pas été utilisé pendant la conception et la sélection du prompt.

Les résultats enregistrés permettront de mesurer :

- l’exactitude des réponses ;
- la validité du format ;
- la latence ;
- le taux d’erreurs techniques ;
- la confiance déclarée ;
- les désaccords entre modèles ;
- les erreurs produites avec une confiance élevée.

In [42]:
from pathlib import Path
import os
import re
import time

import pandas as pd
from dotenv import load_dotenv

In [56]:
# importation des fonctions crées à partir du notebook 03_evaluation_finale stockées dans le dossier src
from src.prompt import prepare_row
from src.llm_inference import call_llm

ImportError: cannot import name 'prepare_row' from 'src.prompt' (C:\Users\MANEL\Dropbox\projet_evaluation_LLM_medicale\src\prompt.py)

In [44]:
# chemin vers le répertoire racine du projet
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "results"
    / "final_evaluation"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRAIN_DATA_PATH = (
    PROCESSED_DIR
    / "benchmark_mcqu_train.parquet"
)

In [45]:
df_trn = pd.read_parquet(
    TRAIN_DATA_PATH
)

In [50]:
RANDOM_SEED = 42
PROMPT_SAMPLE_SIZE = 5
df_sample = (
    df_trn.sample(
        n=PROMPT_SAMPLE_SIZE,
        random_state=RANDOM_SEED
    )
)

In [47]:
# Chargement de la clé API du fichier .env
from google import genai
from openai import OpenAI
env_path = Path.cwd()/ ".env"
load_dotenv(env_path)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Initialisation des clients Gemni et OpenAI
gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

openai_client = OpenAI(
    api_key=OPENAI_API_KEY
)


In [48]:
df_prepared = build_prompt(
    df_sample
)

In [52]:
display(df_prepared)

"Vous devez répondre à une question médicale à choix unique.\n\n             sample_id     id configuration  split  \\\n9571  mcqu_train_13934  13934          mcqu  train   \n9502  mcqu_train_18036  18036          mcqu  train   \n3379   mcqu_train_9861   9861          mcqu  train   \n5407   mcqu_train_1065   1065          mcqu  train   \n7815  mcqu_train_27194  27194          mcqu  train   \n\n                                          clinical_case  \\\n9571                                                      \n9502  Une femme de 47 ans, nulligeste, jusque là nor...   \n3379                                                      \n5407                                                      \n7815  Patiente âgée de 62 ans, sans antécédents path...   \n\n                                               question  \\\n9571  Dans un test de dépistage d'une maladie, la pr...   \n9502  Parmi les examens complémentaires suivants, qu...   \n3379  Lors de la prescription d'un collyre mydriatiq...   \

In [53]:
print(type(df_prepared))

<class 'str'>
